# 1. Header, Citations, Purpose & Topic Bridges

### Theoretical Significance & Research Lineage
3D spatial geometry requires models to respect physical isometry groups ($E(n)$, $SE(3)$). Satorras, Hoogeboom, & Welling (2021) introduced **EGNN** (E(n) Equivariant Graph Neural Networks), establishing that radial coordinate updates achieve exact rotation and translation equivariance without costly spherical harmonics. Qi et al. (2017) pioneered **PointNet**, formalizing continuous permutation-invariant set processing. Bodnar et al. (2021) expanded topological deep learning with **MPSN** (Message Passing Simplicial Networks), passing messages across simplicial complexes to shatter the 1-WL expressivity barrier.

### Purpose & Target Architecture
This notebook constructs from first principles:
1. EGNN layer with $E(n)$-equivariant 3D coordinate updates and invariant scalar messages.
2. Strict mathematical verification of rotational and translational equivariance under arbitrary $SO(3)$ matrices.
3. PointNet permutation-invariant global feature pooling.

# 2. Mathematical Foundations & Governing Equations

### E(n) Equivariance Definition
For orthogonal transformation $R \in O(3)$ and translation $t \in \mathbb{R}^3$:
$$f(R x + t) = R f(x) + t$$

### EGNN Layer Formulation
1. Pairwise invariant Euclidean distance: $d_{ij}^2 = \|x_i - x_j\|^2$.
2. Invariant message: $m_{ij} = \phi_m(h_i, h_j, d_{ij}^2)$.
3. Equivariant coordinate update along relative displacement:
$$x_i^{(l+1)} = x_i^{(l)} + \sum_{j \neq i} (x_i^{(l)} - x_j^{(l)}) \phi_x(m_{ij})$$
4. Invariant feature update: $h_i^{(l+1)} = h_i^{(l)} + \phi_h(h_i^{(l)}, \sum_j m_{ij})$.

### PointNet Set Invariance
Permutation invariance is achieved via shared projection and symmetric max reduction:
$$f(\{p_1, \dots, p_N\}) = \gamma\left(\max_{i=1 \dots N} \text{MLP}(p_i)\right)$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Molecule-like point cloud: N=4 atoms in 3D space, scalar features D=4
N, D = 4, 4
x_coords = torch.randn(N, 3)
h_feats = torch.randn(N, D)
adj = torch.tensor([
    [0.0, 1.0, 1.0, 0.0],
    [1.0, 0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0, 0.0],
], dtype=torch.float32)

print(f"Setup complete: N={N} atoms, initial coordinates shape={x_coords.shape}, features shape={h_feats.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity

def egnn_layer(x: torch.Tensor, h: torch.Tensor, adj: torch.Tensor, w_m: torch.Tensor, b_m: torch.Tensor, w_x: torch.Tensor, b_x: torch.Tensor, w_h: torch.Tensor, b_h: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    n, d = h.shape
    x_i = x.unsqueeze(1)
    x_j = x.unsqueeze(0)
    delta_x = x_i - x_j
    d_sq = torch.sum(delta_x ** 2, dim=-1, keepdim=True)

    h_i = h.unsqueeze(1).expand(n, n, d)
    h_j = h.unsqueeze(0).expand(n, n, d)
    m_in = torch.cat([h_i, h_j, d_sq], dim=-1)

    m_ij = F.silu(m_in @ w_m + b_m) * adj.unsqueeze(-1)
    coord_weights = m_ij @ w_x + b_x
    x_next = x + torch.sum(delta_x * coord_weights, dim=1)

    m_i = torch.sum(m_ij, dim=1)
    h_next = h + F.silu(torch.cat([h, m_i], dim=-1) @ w_h + b_h)
    return x_next, h_next

def pointnet_global_pool(x: torch.Tensor, w_p: torch.Tensor, b_p: torch.Tensor) -> torch.Tensor:
    f = F.relu(x @ w_p + b_p)
    return torch.max(f, dim=0, keepdim=True)[0]

D_m = 4
w_m = torch.randn(2 * D + 1, D_m) * 0.2
b_m = torch.randn(D_m) * 0.1
w_x = torch.randn(D_m, 1) * 0.2
b_x = torch.randn(1) * 0.1
w_h = torch.randn(D + D_m, D) * 0.2
b_h = torch.randn(D) * 0.1

x_next, h_next = egnn_layer(x_coords, h_feats, adj, w_m, b_m, w_x, b_x, w_h, b_h)
print(f"EGNN step complete: x_next shape={x_next.shape}, h_next shape={h_next.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification

# Invariant 1: Exact E(3) Equivariance under 3D Rotation and Translation
# Construct random 3D rotation matrix R via QR decomposition
q_mat, _ = torch.linalg.qr(torch.randn(3, 3))
R = q_mat * torch.det(q_mat)  # Ensure det(R) = +1 (SO(3))
t = torch.tensor([2.5, -1.8, 4.2])

# Transform inputs: x_rot = x @ R.T + t
x_trans = x_coords @ R.T + t
x_trans_next, h_trans_next = egnn_layer(x_trans, h_feats, adj, w_m, b_m, w_x, b_x, w_h, b_h)

# Expected transformed outputs
expected_x_next = x_next @ R.T + t
coord_error = torch.max(torch.abs(x_trans_next - expected_x_next)).item()
feat_error = torch.max(torch.abs(h_trans_next - h_next)).item()

assert coord_error < 1e-5, f"EGNN coordinate equivariance violated: error {coord_error:.2e}"
assert feat_error < 1e-5, f"EGNN feature invariance violated: error {feat_error:.2e}"

# Invariant 2: PointNet permutation invariance
perm = torch.randperm(N)
x_perm = x_coords[perm]
w_p = torch.randn(3, 8) * 0.2
b_p = torch.randn(8) * 0.1
p1 = pointnet_global_pool(x_coords, w_p, b_p)
p2 = pointnet_global_pool(x_perm, w_p, b_p)
perm_error = torch.max(torch.abs(p1 - p2)).item()
assert perm_error < 1e-6, f"PointNet permutation invariance failed: error {perm_error:.2e}"

print(f"All symmetry invariants passed successfully! Coord equivariance error: {coord_error:.2e}, PointNet perm error: {perm_error:.2e}")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

# Benchmark EGNN pairwise update latency
start = time.perf_counter()
for _ in range(500):
    _ = egnn_layer(x_coords, h_feats, adj, w_m, b_m, w_x, b_x, w_h, b_h)
egnn_time = (time.perf_counter() - start) / 500.0

start = time.perf_counter()
for _ in range(500):
    _ = pointnet_global_pool(x_coords, w_p, b_p)
pointnet_time = (time.perf_counter() - start) / 500.0

print(f"EGNN pairwise equivariant step latency: {egnn_time*1e6:.1f} µs")
print(f"PointNet global max pooling latency: {pointnet_time*1e6:.1f} µs")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation

# Pathological Misuse: Injecting absolute coordinates x_i directly into message MLP
def non_equivariant_layer_bad(x: torch.Tensor, h: torch.Tensor, adj: torch.Tensor):
    # Using absolute x instead of pairwise difference delta_x
    bad_messages = x @ torch.randn(3, 3) * 0.1
    return x + bad_messages

x_bad_orig = non_equivariant_layer_bad(x_coords, h_feats, adj)
x_bad_trans = non_equivariant_layer_bad(x_trans, h_feats, adj)

# Check equivariance on bad implementation
bad_expected = x_bad_orig @ R.T + t
bad_error = torch.max(torch.abs(x_bad_trans - bad_expected)).item()

print(f"Pathological (absolute coordinate) equivariance error: {bad_error:.4f} (Violates physical symmetry)")
print(f"Corrected (EGNN radial vector field) equivariance error: {coord_error:.8f} (Exact mathematical symmetry)")

# 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

### Core Architectural Insights
1. **Radial Relative Vectors Guarantee Equivariance**: By multiplying scalar invariant functions $\phi_x(m_{ij})$ exclusively with relative vectors $\Delta x_{ij}$, rotation matrices commute through the update algebraically.
2. **No Clebsch-Gordan Tensor Products Needed**: Unlike earlier SE(3)-Transformers that required spherical harmonic expansions, EGNN achieves geometric equivariance using simple Euclidean norm inputs and scalar MLPs.
3. **PointNet as Set Function**: Permutation-symmetric pooling ($\max$) compresses variable-length point clouds into global descriptors without imposing unnatural 1D sequential order.